# نوت‌بوک ۳ — آزمایش حذفی تک‌مولدی

**هدف:** آموزش سه طبقه‌بند جداگانه، هر یک تنها با داده یک مولد + همه متن‌های انسانی. سپس ارزیابی هر طبقه‌بند روی کل داده آزمون با تفکیک نتیجه به ازای هر مولد.

**پاسخ به پرسش:** آیا طبقه‌بندی که تنها یک مولد را دیده، می‌تواند مولد دیده‌نشده را تشخیص دهد؟

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
DATA_DIR = DRIVE_ROOT / 'data' / 'splits'
ABLATION_DIR = DRIVE_ROOT / 'ablation_train_on_one'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_full = load_jsonl(DATA_DIR / 'train.jsonl')
val_full   = load_jsonl(DATA_DIR / 'val.jsonl')
test_full  = load_jsonl(DATA_DIR / 'test.jsonl')
print(f'train={len(train_full)} val={len(val_full)} test={len(test_full)}')

In [ ]:
# تابع فیلتر: متن‌های انسانی + متن‌های یک مولد خاص
def filter_for_generator(records, gen_name):
    out = []
    for r in records:
        if r['label'] == 0:
            out.append(r)
        elif r.get('model') == gen_name:
            out.append(r)
    return out

GENERATORS = ['deepseek-v3.1', 'gpt-4o-mini', 'gemini-2.5-flash-lite']
GEN_SHORT = {'deepseek-v3.1': 'deepseek', 'gpt-4o-mini': 'gpt', 'gemini-2.5-flash-lite': 'gemini'}

for g in GENERATORS:
    tr = filter_for_generator(train_full, g)
    va = filter_for_generator(val_full, g)
    n_ai = sum(1 for x in tr if x['label'] == 1)
    n_hu = sum(1 for x in tr if x['label'] == 0)
    print(f'{g}: train={len(tr)} (انسان {n_hu}، AI {n_ai}), val={len(va)}')

In [ ]:
# آموزش یک مدل برای هر مولد — نسبت کلاس ۱:۱ پس وزن یکنواخت
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import torch

MODEL_NAME = 'HooshvareLab/bert-fa-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
for g in GENERATORS:
    short = GEN_SHORT[g]
    out_dir = ABLATION_DIR / f'trained_on_{short}'
    if (out_dir / 'test_predictions.jsonl').exists():
        print(f'[{g}] قبلاً انجام شده — رد می‌شویم')
        continue
    print(f'\n=== آموزش روی {g} ===')
    tr = filter_for_generator(train_full, g)
    va = filter_for_generator(val_full, g)

    ds_tr = Dataset.from_list(tr).map(tokenize, batched=True)
    ds_va = Dataset.from_list(va).map(tokenize, batched=True)
    ds_te = Dataset.from_list(test_full).map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=str(out_dir / 'tmp'),
        num_train_epochs=3, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        eval_strategy='epoch', save_strategy='no', logging_steps=50, report_to=[]
    )
    trainer = WeightedTrainer(
        class_weights=torch.tensor([1.0, 1.0]),
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=metrics
    )
    trainer.train()

    # پیش‌بینی روی کل آزمون
    pred_obj = trainer.predict(ds_te)
    preds = np.argmax(pred_obj.predictions, axis=-1)
    truth = pred_obj.label_ids
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'test_predictions.jsonl', 'w', encoding='utf-8') as f:
        for r, p, t in zip(test_full, preds, truth):
            f.write(json.dumps({
                'id': r['id'],
                'source': r.get('source', 'human'),
                'model': r.get('model', 'human'),
                'true_label': int(t), 'pred_label': int(p),
                'correct': bool(p == t)
            }, ensure_ascii=False) + '\n')
    print(f'  ✓ {len(test_full)} پیش‌بینی ذخیره شد در {out_dir}')

In [ ]:
# ساخت ماتریس نتایج
matrix = {}
for g in GENERATORS:
    short = GEN_SHORT[g]
    preds = [json.loads(l) for l in open(ABLATION_DIR / f'trained_on_{short}' / 'test_predictions.jsonl', encoding='utf-8') if l.strip()]
    overall = sum(p['correct'] for p in preds) / len(preds)
    per_gen = {}
    for tg in GENERATORS:
        sub = [p for p in preds if p['model'] == tg]
        per_gen[tg] = sum(p['correct'] for p in sub) / len(sub) if sub else 0
    matrix[f'trained_on_{short}'] = {'overall': overall, 'by_test_generator': per_gen}
    print(f'{g}: کل={overall:.4f} | ' + ', '.join(f'{k}={v:.3f}' for k, v in per_gen.items()))

with open(DRIVE_ROOT / 'results' / 'train_on_one_matrix.json', 'w', encoding='utf-8') as f:
    json.dump({'experiment': 'train_on_one_ablation', 'matrix': matrix}, f, ensure_ascii=False, indent=2)
print('\n✓ ماتریس ذخیره شد در results/train_on_one_matrix.json')